# Phase 1 : Extraction et chargement du dataset INSERSUP

**Source** : `Projet/csv/dataset.csv` (export INSERSUP, millésime `2026_S1`)

Ce notebook couvre la phase 1 du projet final : vérifier la source, diagnostiquer sa
structure, charger les données proprement et produire deux jeux de données prêts pour
l'EDA de la phase 2.

**Sommaire**
1. Vérification du fichier source
2. Inventaire des colonnes
3. Diagnostic de remplissage et choix de la colonne cible
4. Niveaux d'agrégation présents dans le fichier
5. Chargement filtré (lecture par blocs)
6. Validation post-extraction
7. Récapitulatif et export

In [1]:
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

# Chemins : résolus depuis l'emplacement du notebook, pas depuis le cwd du kernel.
# Path.cwd() dépend du réglage jupyter.notebookFileRoot et casse dès qu'il change.
RACINE_PROJET = Path.cwd()
if RACINE_PROJET.name != "Projet":
    RACINE_PROJET = RACINE_PROJET / "Projet"

DATA_PATH = RACINE_PROJET / "csv" / "dataset.csv"
SORTIE_MODELISATION = RACINE_PROJET / "csv" / "dataset_phase1_modelisation.csv"
SORTIE_ANALYTIQUE = RACINE_PROJET / "csv" / "dataset_phase1_analytique.csv"

# Paramètres de lecture, déterminés à l'inspection du fichier brut (section 1).
PARAMS_LECTURE = {
    "sep": ";",              # séparateur point-virgule
    "encoding": "utf-8-sig", # BOM UTF-8 en tête de fichier
    "dtype": str,            # tout en texte, conversion explicite ensuite
    "na_values": ["nd", "ns"],  # codes de non-diffusion propres à INSERSUP
}
TAILLE_BLOC = 200_000  # lecture par blocs : le fichier fait 773 Mo

print("Chemin du dataset :", DATA_PATH)
print("Fichier présent   :", DATA_PATH.exists())

Chemin du dataset : /home/baptiste/Documents/master LiveCampus/machine_learning/Projet/csv/dataset.csv
Fichier présent   : True


## 1. Vérification du fichier source

Avant tout chargement pandas, on regarde le fichier brut : taille, encodage,
séparateur, présence d'un en-tête.

In [2]:
taille_mo = DATA_PATH.stat().st_size / 1024**2
print(f"Taille : {taille_mo:.0f} Mo")

with open(DATA_PATH, "rb") as f:
    premiers_octets = f.read(3)
    f.seek(0)
    premiere_ligne = f.readline().decode("utf-8-sig")

print("BOM UTF-8 détecté :", premiers_octets == b"\xef\xbb\xbf")
print("Séparateur ';' :", premiere_ligne.count(";"), "occurrences dans l'en-tête")
print("Séparateur ',' :", premiere_ligne.count(","), "occurrences dans l'en-tête")
print()
print("Premiers champs de l'en-tête :")
print(premiere_ligne.split(";")[:6])

Taille : 738 Mo
BOM UTF-8 détecté : True
Séparateur ';' : 100 occurrences dans l'en-tête
Séparateur ',' : 0 occurrences dans l'en-tête

Premiers champs de l'en-tête :
['Diffusion des données', 'Région', 'Académie', 'Établissement', 'Type de diplôme', 'Domaine disciplinaire']


Le BOM impose `encoding='utf-8-sig'`. Sans lui, la première colonne s'appellerait
`\ufeffDiffusion des données` et tout appel par son nom échouerait.

## 2. Inventaire des colonnes

101 colonnes : on les regroupe par famille pour s'y retrouver plutôt que de les lister à plat.

In [3]:
colonnes = pd.read_csv(DATA_PATH, nrows=0, **PARAMS_LECTURE).columns.tolist()
print(f"{len(colonnes)} colonnes\n")

familles = {
    "Identification / géographie": [c for c in colonnes if c in (
        "Diffusion des données", "Région", "Académie", "Établissement", "Code de la région",
        "Code de l'académie", "Identifiant interne de l'établissement",
        "Identifiant interne de l'établissement actuel", "Établissement actuel",
        "Code UAI de l'établissement", "Établissement dans les données sources")],
    "Formation": [c for c in colonnes if c in (
        "Type de diplôme", "Domaine disciplinaire", "Discipline", "Secteur disciplinaire",
        "Libellé du diplôme", "type_diplome", "Code du domaine disciplinaire",
        "Code de la discipline", "Code du secteur disciplinaire", "Code du diplôme SISE")],
    "Axes d'agrégation": ["Genre", "Nationalité", "Régime d'inscription", "Promotion",
                          "Obtention du diplôme", "Source de données"],
    "Qualité (flags)": [c for c in colonnes if "Flag" in c or "Exception" in c],
    "Effectifs": [c for c in colonnes if "Nombre de" in c],
    "Taux": [c for c in colonnes if "Taux" in c],
    "Salaires": [c for c in colonnes if "salaire" in c.lower()],
}
familles["Autres"] = [c for c in colonnes if not any(c in v for v in familles.values())]

for nom, cols in familles.items():
    print(f"{nom:30} {len(cols):>3} colonnes")

101 colonnes

Identification / géographie     11 colonnes
Formation                       10 colonnes
Axes d'agrégation                6 colonnes
Qualité (flags)                 10 colonnes
Effectifs                       24 colonnes
Taux                            21 colonnes
Salaires                        15 colonnes
Autres                           4 colonnes


## 3. Diagnostic de remplissage et choix de la colonne cible

Une colonne présente dans l'en-tête n'est pas une colonne remplie. On mesure le taux
de remplissage réel de toutes les colonnes numériques avant de figer la cible.

Cette lecture parcourt les 773 Mo par blocs : compter 2 à 4 minutes.

In [4]:
colonnes_num = [c for c in colonnes
                if "Nombre de" in c or "Taux" in c or "salaire" in c.lower()]

remplissage = dict.fromkeys(colonnes_num, 0)
n_lignes = 0

for bloc in pd.read_csv(DATA_PATH, usecols=colonnes_num, chunksize=TAILLE_BLOC, **PARAMS_LECTURE):
    n_lignes += len(bloc)
    for col in colonnes_num:
        remplissage[col] += pd.to_numeric(bloc[col], errors="coerce").notna().sum()

diagnostic = (
    pd.Series(remplissage, name="valeurs_numeriques")
    .to_frame()
    .assign(taux_remplissage=lambda d: (d["valeurs_numeriques"] / n_lignes * 100).round(2))
    .sort_values("taux_remplissage", ascending=False)
)

print(f"{n_lignes:,} lignes analysées\n".replace(",", " "))
print("--- Colonnes entièrement vides ---")
print(diagnostic[diagnostic["taux_remplissage"] == 0].index.tolist())

1 036 781 lignes analysées

--- Colonnes entièrement vides ---
['6-1er quartile du salaire mensuel net en équivalent temps plein - 6 mois après le diplôme', '6-3ème quartile du salaire mensuel net en équivalent temps plein - 6 mois après le diplôme', '6-Salaire mensuel net médian en équivalent temps plein - 6 mois après le diplôme', "6-Taux d'emploi - 6 mois après le diplôme", "12-Taux d'emploi - 12 mois après le diplôme", "18-Taux d'emploi - 18 mois après le diplôme", "6-Taux de sortants en emploi à l'étranger - 6 mois après le diplôme", "12-Taux de sortants en emploi à l'étranger - 12 mois après le diplôme", "18-Taux de sortants en emploi à l'étranger - 18 mois après le diplôme"]


In [5]:
# Les colonnes de taux les mieux remplies, candidates au rôle de cible
diagnostic[diagnostic.index.str.contains("Taux")].head(12)

,valeurs_numeriques,taux_remplissage
18-Taux d'emploi salarié en France - 18 mois après le diplôme,439935,42.43
12-Taux d'emploi salarié en France - 12 mois après le diplôme,439935,42.43
12-Taux de sortants en emploi non salarié - 12 mois après le diplôme,439935,42.43
18-Taux de sortants en emploi non salarié - 18 mois après le diplôme,439935,42.43
18-Taux de sortants en emploi stable - 18 mois après le diplôme,439935,42.43
12-Taux de sortants en emploi stable - 12 mois après le diplôme,439935,42.43
6-Taux de sortants en emploi non salarié - 6 mois après le diplôme,438696,42.31
6-Taux d'emploi salarié en France - 6 mois après le diplôme,438696,42.31
6-Taux de sortants en emploi stable - 6 mois après le diplôme,438696,42.31
24-Taux d'emploi salarié en France - 24 mois après le diplôme,353656,34.11


### Conclusion : changement de colonne cible

La cible retenue au cadrage (étape 0) était `6-Taux d'emploi - 6 mois après le diplôme`.
Le diagnostic ci-dessus montre qu'elle est **vide sur les 1 036 781 lignes** de ce
millésime, comme ses variantes à 12 et 18 mois. Elle est inutilisable.

**Nouvelle cible** : `6-Taux d'emploi salarié en France - 6 mois après le diplôme`,
remplie sur 42,3 % des lignes (438 696 valeurs). C'est la mesure d'emploi à 6 mois la
mieux couverte du fichier, et elle répond à la même question prédictive.

In [6]:
CIBLE = "6-Taux d'emploi salarié en France - 6 mois après le diplôme"

print("Cible abandonnée :")
print(diagnostic.loc["6-Taux d'emploi - 6 mois après le diplôme"].to_string())
print("\nCible retenue :")
print(diagnostic.loc[CIBLE].to_string())

Cible abandonnée :
valeurs_numeriques    0.0
taux_remplissage      0.0

Cible retenue :
valeurs_numeriques    438696.00
taux_remplissage          42.31


## 4. Niveaux d'agrégation présents dans le fichier

INSERSUP n'est pas une table de faits plate. C'est un cube **avec ses marges** : chaque
axe contient à la fois ses modalités et une ligne de total. Trois familles de marges
cohabitent dans le même fichier.

**Marges démographiques** : `Genre`, `Nationalité`, `Régime d'inscription` et
`Obtention du diplôme` ont chacun une modalité `ensemble` qui totalise les autres.

**Marges temporelles** : `Promotion` mélange des années simples (`2019`) et des cumuls de
deux promotions (`2019,2020`), ces derniers utilisés quand l'effectif est trop faible
pour publier une année seule.

**Marges géographiques et disciplinaires** : des lignes `National` agrègent tous les
établissements, et `Tous domaines disciplinaires` / `Toutes disciplines` /
`Tous secteurs disciplinaires` agrègent toutes les formations.

Conséquence : une même cohorte est comptée plusieurs fois. Entraîner un modèle sur le
fichier brut reviendrait à apprendre sur des lignes qui recomptent leurs propres
sous-lignes, avec en prime une fuite de la cible (le total contient l'information de ses
composantes).

In [7]:
AXES_AGREGATION = ["Genre", "Nationalité", "Régime d'inscription",
                   "Obtention du diplôme", "Promotion", "Source de données"]

apercu = pd.read_csv(DATA_PATH, usecols=AXES_AGREGATION, nrows=300_000, **PARAMS_LECTURE)
for col in AXES_AGREGATION:
    print(f"--- {col} ---")
    print(apercu[col].value_counts(dropna=False).head(11).to_string())
    print()

--- Genre ---
Genre
ensemble    104878
femme        97587
homme        97535

--- Nationalité ---
Nationalité
ensemble    151433
français    148567

--- Régime d'inscription ---
Régime d'inscription
ensemble         206374
apprentissage     93626

--- Obtention du diplôme ---
Obtention du diplôme
ensemble    152239
diplômé     147761

--- Promotion ---
Promotion
2019         44319
2022,2023    33217
2021,2022    32298
2023,2024    31359
2020,2021    28688
2019,2020    23865
2020         22184
2021         22065
2024         21272
2022         20810
2023         19923

--- Source de données ---
Source de données
insersup        299114
IP augmentée       886



In [8]:
# Marges géographiques et disciplinaires : elles se lisent dans les valeurs, pas dans
# une colonne dédiée.
COLS_MARGE = ["Région", "Établissement", "Domaine disciplinaire",
              "Discipline", "Secteur disciplinaire"]

apercu_marges = pd.read_csv(DATA_PATH, usecols=COLS_MARGE, nrows=300_000, **PARAMS_LECTURE)
for col in COLS_MARGE:
    modalites = apercu_marges[col].dropna().unique()
    totaux = [m for m in modalites if m.lower().startswith(("national", "tous ", "toutes "))]
    print(f"{col:24} {len(modalites):>4} modalités | valeurs de total : {totaux}")

Région                     21 modalités | valeurs de total : ['National']
Établissement             410 modalités | valeurs de total : ['National']
Domaine disciplinaire       5 modalités | valeurs de total : ['Tous domaines disciplinaires']
Discipline                 16 modalités | valeurs de total : ['Toutes disciplines']
Secteur disciplinaire      54 modalités | valeurs de total : ['Tous secteurs disciplinaires']


`Source de données` compte deux modalités (`insersup` et `IP augmentée`), mais les 886
lignes `IP augmentée` n'ont pas de cible renseignée : la colonne est donc constante sur
le périmètre retenu et n'apporte rien au modèle. Elle est écartée.

**Décision de granularité** : deux jeux, un par usage.

| Jeu produit | Filtre | Usage |
|---|---|---|
| `dataset_phase1_modelisation.csv` | cible renseignée · `Genre`, `Nationalité`, `Régime d'inscription` = `ensemble` · `Obtention du diplôme` = `diplômé` · promotion simple · **aucune marge géographique ni disciplinaire** | Phase 7 : une ligne = une formation × promotion, sans double comptage |
| `dataset_phase1_analytique.csv` | cible renseignée, tous niveaux conservés mais **marqués par des indicateurs explicites** | Phases 2 et 5 : l'EDA a besoin des marges, notamment pour la question business 3 (genre / nationalité / régime) |

Le jeu analytique garde les marges parce qu'elles sont utiles en analyse descriptive ;
il les signale par les colonnes `ligne_agregee`, `promotion_cumulee` et `marge_geo_disc`
pour qu'aucune agrégation ne soit mélangée à un détail par erreur.

## 5. Chargement filtré

Lecture par blocs avec conversion des types et filtrage à la volée : le fichier complet
ne tient pas confortablement en mémoire.

In [9]:
DIMENSIONS = ["Région", "Académie", "Établissement", "Code UAI de l'établissement",
              "Type de diplôme", "Domaine disciplinaire", "Discipline",
              "Secteur disciplinaire", "Libellé du diplôme"]
# Le code UAI est l'identifiant national des établissements : c'est la clé de jointure
# vers toute source d'enrichissement. Il vaut 'all' sur les lignes d'agrégat national.

AXES = ["Genre", "Nationalité", "Régime d'inscription", "Obtention du diplôme", "Promotion"]

QUALITE = ["6-Flag - 6 mois après le diplôme", "6-Exception - 6 mois après le diplôme"]

MESURES = [
    "6-Nombre de sortants - 6 mois après le diplôme",
    "6-Nombre de poursuivants - 6 mois après le diplôme",
    CIBLE,
    "6-Nombre de sortants en emploi stable - 6 mois après le diplôme",
    "6-Taux de sortants en emploi stable - 6 mois après le diplôme",
    "6-Nombre de sortants en emploi non salarié - 6 mois après le diplôme",
    "6-Taux de sortants en emploi non salarié - 6 mois après le diplôme",
    # autres horizons, pour la question business 5 (évolution dans le temps)
    "12-Taux d'emploi salarié en France - 12 mois après le diplôme",
    "18-Taux d'emploi salarié en France - 18 mois après le diplôme",
    "24-Taux d'emploi salarié en France - 24 mois après le diplôme",
    "30-Taux d'emploi salarié en France - 30 mois après le diplôme",
]

COLONNES_RETENUES = DIMENSIONS + AXES + QUALITE + MESURES
print(f"{len(COLONNES_RETENUES)} colonnes retenues sur {len(colonnes)}")

# Toutes les colonnes absentes du fichier seraient une erreur de nommage : on le vérifie
# au lieu de les filtrer en silence.
manquantes = [c for c in COLONNES_RETENUES if c not in colonnes]
assert not manquantes, f"Colonnes introuvables dans le CSV : {manquantes}"
print("Toutes les colonnes demandées existent bien dans le fichier.")

27 colonnes retenues sur 101
Toutes les colonnes demandées existent bien dans le fichier.


In [10]:
def preparer(bloc: pd.DataFrame) -> pd.DataFrame:
    """Convertit les mesures en numérique et ajoute les indicateurs de niveau."""
    bloc = bloc.copy()
    for col in MESURES:
        bloc[col] = pd.to_numeric(bloc[col], errors="coerce")

    bloc["promotion_cumulee"] = bloc["Promotion"].str.contains(",", na=False)
    bloc["promotion_debut"] = pd.to_numeric(
        bloc["Promotion"].str.split(",").str[0], errors="coerce"
    ).astype("Int64")
    bloc["ligne_agregee"] = (
        (bloc["Genre"] == "ensemble")
        & (bloc["Nationalité"] == "ensemble")
        & (bloc["Régime d'inscription"] == "ensemble")
    )
    bloc["marge_geo_disc"] = (
        bloc["Région"].eq("National")
        | bloc["Domaine disciplinaire"].eq("Tous domaines disciplinaires")
        | bloc["Discipline"].eq("Toutes disciplines")
        | bloc["Secteur disciplinaire"].eq("Tous secteurs disciplinaires")
    )
    return bloc


morceaux_analytique = []
n_brut = 0

for bloc in pd.read_csv(DATA_PATH, usecols=COLONNES_RETENUES,
                        chunksize=TAILLE_BLOC, **PARAMS_LECTURE):
    n_brut += len(bloc)
    bloc = preparer(bloc)
    # une ligne sans cible n'apporte rien, ni à l'EDA de la cible ni au modèle
    morceaux_analytique.append(bloc[bloc[CIBLE].notna()])

df_analytique = pd.concat(morceaux_analytique, ignore_index=True)
del morceaux_analytique

print(f"Lignes lues            : {n_brut:>9,}".replace(",", " "))
print(f"Lignes avec cible      : {len(df_analytique):>9,}".replace(",", " "))
print(f"Perte (cible absente)  : {n_brut - len(df_analytique):>9,}".replace(",", " "))

Lignes lues            : 1 036 781
Lignes avec cible      :   438 696
Perte (cible absente)  :   598 085


In [11]:
etapes = {
    "cible renseignée": pd.Series(True, index=df_analytique.index),
    "+ démographie = ensemble": df_analytique["ligne_agregee"],
    "+ obtention = diplômé": df_analytique["Obtention du diplôme"] == "diplômé",
    "+ promotion simple": ~df_analytique["promotion_cumulee"],
    "+ hors marges géo/disc": ~df_analytique["marge_geo_disc"],
}

masque_modelisation = pd.Series(True, index=df_analytique.index)
for libelle, condition in etapes.items():
    masque_modelisation &= condition
    print(f"{libelle:28} {masque_modelisation.sum():>7,} lignes".replace(",", " "))

# Colonnes devenues inutiles à ce niveau : les axes filtrés (constants), les indicateurs
# de niveau, et les flags qualité qui ne concernent que les promotions cumulées.
df_modelisation = (
    df_analytique[masque_modelisation]
    .drop(columns=["ligne_agregee", "promotion_cumulee", "marge_geo_disc",
                   "Genre", "Nationalité", "Régime d'inscription", "Obtention du diplôme",
                   *QUALITE])
    .reset_index(drop=True)
)

print(f"Jeu analytique   : {df_analytique.shape[0]:>7,} lignes × {df_analytique.shape[1]:>2} colonnes"
      .replace(",", " "))
print(f"Jeu modélisation : {df_modelisation.shape[0]:>7,} lignes × {df_modelisation.shape[1]:>2} colonnes"
      .replace(",", " "))

cible renseignée             438 696 lignes
+ démographie = ensemble      78 535 lignes
+ obtention = diplômé         37 472 lignes
+ promotion simple            27 149 lignes
+ hors marges géo/disc        17 065 lignes
Jeu analytique   : 438 696 lignes × 31 colonnes
Jeu modélisation :  17 065 lignes × 22 colonnes


Les colonnes `6-Flag` et `6-Exception` sont retirées du jeu de modélisation : elles
signalent le cumul de deux promotions pour cause d'effectif insuffisant, or ce cas vient
d'être exclu. Elles restent présentes dans le jeu analytique, où les promotions cumulées
sont conservées.

## 6. Validation post-extraction

Les cinq réflexes demandés par le guide : dimensions, types, aperçu, valeurs manquantes,
unicité.

In [12]:
df_modelisation.info()

<class 'pandas.DataFrame'>
RangeIndex: 17065 entries, 0 to 17064
Data columns (total 22 columns):
 #   Column                                                                Non-Null Count  Dtype  
---  ------                                                                --------------  -----  
 0   Région                                                                17065 non-null  str    
 1   Académie                                                              17065 non-null  str    
 2   Établissement                                                         17065 non-null  str    
 3   Type de diplôme                                                       17065 non-null  str    
 4   Domaine disciplinaire                                                 17065 non-null  str    
 5   Discipline                                                            17065 non-null  str    
 6   Secteur disciplinaire                                                 17065 non-null  str    
 7   Libell

In [13]:
print("--- Distribution de la cible (jeu de modélisation) ---")
print(df_modelisation[CIBLE].describe().round(2).to_string())
print()
print("--- Valeurs manquantes (top 10) ---")
manquants = (df_modelisation.isna().mean() * 100).round(1).sort_values(ascending=False)
print(manquants[manquants > 0].head(10).to_string())

--- Distribution de la cible (jeu de modélisation) ---
count    17065.00
mean        55.89
std         18.47
min          0.00
25%         43.24
50%         56.10
75%         69.23
max        100.00

--- Valeurs manquantes (top 10) ---
24-Taux d'emploi salarié en France - 24 mois après le diplôme    17.2
30-Taux d'emploi salarié en France - 30 mois après le diplôme    17.2


In [14]:
# Clé primaire : le fichier n'en fournit pas (la colonne `id` d'origine est vide).
# On vérifie si la combinaison de dimensions identifie bien une ligne unique.
CLE = ["Code UAI de l'établissement", "Type de diplôme", "Domaine disciplinaire",
       "Discipline", "Secteur disciplinaire", "Libellé du diplôme", "Promotion"]

n_doublons = df_modelisation.duplicated(subset=CLE).sum()
print(f"Doublons sur la clé métier      : {n_doublons}")
print(f"Lignes strictement identiques   : {df_modelisation.duplicated().sum()}")

Doublons sur la clé métier      : 107
Lignes strictement identiques   : 0


In [15]:
# À quoi ressemblent ces doublons ? Deux lignes de même clé mais de cible différente
# signalent une clé incomplète, pas une duplication à supprimer.
if n_doublons:
    exemples = df_modelisation[df_modelisation.duplicated(subset=CLE, keep=False)]
    print(f"{len(exemples)} lignes concernées par un conflit de clé")
    groupes = exemples.groupby(CLE, dropna=False)[CIBLE].nunique()
    print(f"  groupes à cible identique  : {(groupes == 1).sum()}  (vraie duplication)")
    print(f"  groupes à cible différente : {(groupes > 1).sum()}  (clé incomplète)")
    print()
    print(exemples.sort_values(CLE).head(4)[
        ["Établissement", "Libellé du diplôme", "Promotion",
         "6-Nombre de sortants - 6 mois après le diplôme", CIBLE]
    ].to_string(index=False))

201 lignes concernées par un conflit de clé
  groupes à cible identique  : 0  (vraie duplication)
  groupes à cible différente : 94  (clé incomplète)

         Établissement                              Libellé du diplôme Promotion  6-Nombre de sortants - 6 mois après le diplôme  6-Taux d'emploi salarié en France - 6 mois après le diplôme
Université Côte d'Azur DROIT, ECONOMIE, GESTION : FINANCE COMPTABILITE      2019                                            24.0                                                        58.33
Université Côte d'Azur DROIT, ECONOMIE, GESTION : FINANCE COMPTABILITE      2019                                            20.0                                                        40.00
Université Côte d'Azur DROIT, ECONOMIE, GESTION : FINANCE COMPTABILITE      2019                                            33.0                                                        72.73
Université Côte d'Azur           DROIT, ECONOMIE, GESTION : MANAGEMENT      2019         

Le traitement de ces conflits relève de la phase 3 (nettoyage). Ce qui compte ici est de
les avoir détectés et de savoir que la clé métier retenue ne suffit pas encore à
identifier une ligne de façon unique.

In [16]:
# Effectifs : un taux calculé sur 3 sortants n'a pas de valeur statistique.
effectif = "6-Nombre de sortants - 6 mois après le diplôme"
print(df_modelisation[effectif].describe().round(1).to_string())
print()
for seuil in (10, 20, 30, 50):
    n = (df_modelisation[effectif] >= seuil).sum()
    print(f"lignes avec au moins {seuil:>2} sortants : {n:>6} ({n / len(df_modelisation):.1%})")

count    17065.0
mean        62.2
std         92.8
min         20.0
25%         26.0
50%         37.0
75%         62.0
max       1882.0

lignes avec au moins 10 sortants :  17065 (100.0%)
lignes avec au moins 20 sortants :  17065 (100.0%)
lignes avec au moins 30 sortants :  11074 (64.9%)
lignes avec au moins 50 sortants :   5854 (34.3%)


Le filtrage sur un effectif minimal relève de la phase 3 (nettoyage) : on le documente
ici sans l'appliquer, pour garder la phase 1 purement descriptive.

In [17]:
df_modelisation.head()

,Région,Académie,Établissement,Type de diplôme,Domaine disciplinaire,Discipline,Secteur disciplinaire,Libellé du diplôme,Promotion,6-Nombre de sortants - 6 mois après le diplôme,6-Nombre de poursuivants - 6 mois après le diplôme,6-Taux d'emploi salarié en France - 6 mois après le diplôme,12-Taux d'emploi salarié en France - 12 mois après le diplôme,18-Taux d'emploi salarié en France - 18 mois après le diplôme,24-Taux d'emploi salarié en France - 24 mois après le diplôme,30-Taux d'emploi salarié en France - 30 mois après le diplôme,6-Nombre de sortants en emploi non salarié - 6 mois après le diplôme,6-Taux de sortants en emploi non salarié - 6 mois après le diplôme,6-Nombre de sortants en emploi stable - 6 mois après le diplôme,6-Taux de sortants en emploi stable - 6 mois après le diplôme,Code UAI de l'établissement,promotion_debut
0,Pays de la Loire,Nantes,École de gestion et de commerce de Vendée,Diplôme visé niveau bac + 3,"Droit, économie, gestion","Sciences économiques, gestion",Sciences de gestion,DIPLOME DE L'ECOLE DE GESTION ET DE COMMERCE D...,2019,22.0,10.0,90.91,90.91,90.91,95.45,77.27,0.0,0.00,6.0,30.00,0851465F,2019
1,Auvergne-Rhône-Alpes,Lyon,EM Lyon Business School,Diplôme visé niveau bac + 5 grade master,"Droit, économie, gestion","Sciences économiques, gestion",Sciences de gestion,DIPLOME DE L'ECOLE DE MANAGEMENT DE LYON (PGE),2020,605.0,325.0,52.89,60.96,63.84,63.04,62.56,0.0,0.00,270.0,84.38,0690197P,2020
2,Île-de-France,Versailles,EDC Paris Business School,Diplôme visé niveau bac + 5 grade master,"Droit, économie, gestion","Sciences économiques, gestion",Sciences de gestion,DIPLOME DE L'ECOLE DES DIRIGEANTS ET CREATEURS...,2024,133.0,9.0,45.11,58.09,51.47,NaN,NaN,0.0,0.00,40.0,66.67,0922007G,2024
3,Île-de-France,Versailles,École des hautes études commerciales de Paris,Diplôme visé niveau bac + 5 grade master,"Droit, économie, gestion","Sciences économiques, gestion",Sciences de gestion,DIPLOME DE L'ECOLE DES HAUTES ETUDES COMMERCIA...,2019,261.0,20.0,35.63,32.95,35.25,36.40,36.78,0.0,0.00,79.0,84.95,0783054W,2019
4,Île-de-France,Versailles,École des hautes études commerciales de Paris,Diplôme visé niveau bac + 5 grade master,"Droit, économie, gestion","Sciences économiques, gestion",Sciences de gestion,DIPLOME DE L'ECOLE DES HAUTES ETUDES COMMERCIA...,2023,623.0,151.0,42.54,54.00,55.40,57.36,55.40,34.0,5.46,221.0,83.40,0783054W,2023


## 7. Récapitulatif et export

In [18]:
recap = pd.DataFrame([
    {
        "Source": "1",
        "Format": "CSV",
        "Fichier": str(DATA_PATH.relative_to(RACINE_PROJET.parent)),
        "Lignes": n_brut,
        "Colonnes": len(colonnes),
        "Observations": "BOM UTF-8, sep=';', manquants codés 'nd'/'ns', 6 colonnes de taux vides",
    },
])
recap

,Source,Format,Fichier,Lignes,Colonnes,Observations
0,1,CSV,Projet/csv/dataset.csv,1036781,101,"BOM UTF-8, sep=';', manquants codés 'nd'/'ns',..."


In [19]:
df_modelisation.to_csv(SORTIE_MODELISATION, index=False, encoding="utf-8")
df_analytique.to_csv(SORTIE_ANALYTIQUE, index=False, encoding="utf-8")

for chemin, df in ((SORTIE_MODELISATION, df_modelisation), (SORTIE_ANALYTIQUE, df_analytique)):
    print(f"{chemin.name:38} {df.shape[0]:>7,} lignes  "
          f"{chemin.stat().st_size / 1024**2:>6.1f} Mo".replace(",", " "))

dataset_phase1_modelisation.csv         17 065 lignes     4.7 Mo
dataset_phase1_analytique.csv          438 696 lignes   147.3 Mo


## Bilan de la phase 1

| Point | Résultat |
|---|---|
| Fichier source | `Projet/csv/dataset.csv`, 773 Mo, 1 036 781 lignes × 101 colonnes |
| Paramètres de lecture | `sep=';'`, `encoding='utf-8-sig'`, `na_values=['nd','ns']`, lecture par blocs de 200 000 |
| Problème majeur | La cible du cadrage était vide à 100 %, remplacée par le taux d'emploi salarié en France à 6 mois |
| Problème structurel | Marges démographiques, temporelles, géographiques et disciplinaires superposées, traitées par séparation en deux jeux |
| Clé primaire | Absente du fichier (`id` vide) ; la clé métier reconstruite laisse des conflits, à traiter en phase 3 |
| Livrables | `dataset_phase1_modelisation.csv` et `dataset_phase1_analytique.csv` |

**Réserve** : le projet ne repose encore que sur une source et un seul format. Le guide
en demande deux au minimum. Piste identifiée : l'API JSON data.gouv.fr qui expose le même
jeu INSERSUP, ou un référentiel d'établissements pour enrichir par la géographie.

Prochaine étape : phase 2, EDA diagnostique sur `dataset_phase1_analytique.csv`.